# LangChain's components

## Models and Prompts

In [1]:


from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama


#from langchain_ollama import OllamaLLM

def call_chatgpt(message):
    # Initialize OpenAI model
    openai_model = ChatOpenAI(
        model="gpt-4o-mini",
        #openai_api_key=os.environ["OPENAI_API_KEY"],
        temperature=0
    )
    test_response = openai_model.invoke(message)
    print(test_response.content)


def call_ollama(message):
    """Initialize Ollama model"""
    ollama_model = ChatOllama(
        model="gpt-oss:20b",  # Change this to your preferred Ollama model
        temperature=0
    )
    test_response = ollama_model.invoke(message)
    print(test_response.content)

def call_claude(message):
    # Initialize Claude local model
    claude_model = ChatAnthropic(
        model="claude-3-5-haiku-20241022",
        #anthropic_api_key = os.getenv('ANTHROPIC_API_KEY'),
        temperature=0
    )
    test_response = claude_model.invoke(message)
    print(test_response.content)


try:
    # Initialize and set up environment
    load_dotenv(override=True)

    message = "tell me a joke in russian language"

    print("Testing OpenAI model...")
    call_chatgpt(message)

    print("\n")

    print("Testing Ollama model...")
    call_ollama(message)

    print("\n")

    print("Testing Claude model...")
    call_claude(message)


except Exception as e:
    print(f"❌ Setup failed: {e}")


Testing OpenAI model...
Конечно! Вот шутка на русском:

Почему программисты не любят природу?

Потому что в ней слишком много багов!


Testing Ollama model...
Почему в России всегда так быстро говорят «бы»?  
Потому что в русском языке «бы» — это не просто частица, а целый язык!


Testing Claude model...
Here's a classic Russian joke in Russian:

Идёт экзамен. Преподаватель спрашивает студента:
- Что такое параллелепипед?
Студент отвечает:
- Это такая штука, которая... ну... как его... параллельная... и пипед!

In English, this translates to:

During an exam, the professor asks a student:
- What is a parallelepiped?
The student answers:
- It's such a thing that... well... how do you say... parallel... and piped!

The humor comes from the student's inability to explain the geometric term and his awkward attempt to break down the word.

Would you like me to explain the joke or tell you another Russian joke?


In [2]:
from langchain_core.prompts import PromptTemplate

template = """Sentence: {sentence}
Translation in {language}:"""
prompt = PromptTemplate(template=template, input_variables=["sentence", "language"])

print(prompt.format(sentence = "the cat is on the table", language = "russian"))

Sentence: the cat is on the table
Translation in russian:


## Data Connections

### Document loaders

In [3]:
import csv

# Sample data
data = [
    ['Name', 'Age', 'City'],
    ['John', 25, 'New York'],
    ['Emily', 28, 'Los Angeles'],
    ['Michael', 22, 'Chicago']
]

# File name
file_name = 'sample.csv'

# Write data to CSV file
with open(file_name, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerows(data)

print(f'Sample CSV file "{file_name}" generated and saved.')



Sample CSV file "sample.csv" generated and saved.


In [1]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path='sample.csv')
data = loader.load()
print(data)

[Document(metadata={'source': 'sample.csv', 'row': 0}, page_content='Name: John\nAge: 25\nCity: New York'), Document(metadata={'source': 'sample.csv', 'row': 1}, page_content='Name: Emily\nAge: 28\nCity: Los Angeles'), Document(metadata={'source': 'sample.csv', 'row': 2}, page_content='Name: Michael\nAge: 22\nCity: Chicago')]


### Document splitters

In [3]:
# Sample sentences about mountains and nature
content = """Amidst the serene landscape, towering mountains stand as majestic guardians of nature's beauty.
The crisp mountain air carries whispers of tranquility, while the rustling leaves compose a symphony of wilderness.
Nature's palette paints the mountains with hues of green and brown, creating an awe-inspiring sight to behold.
As the sun rises, it casts a golden glow on the mountain peaks, illuminating a world untouched and wild."""

# File name
file_name = 'mountain.txt'

# Write content to text file
with open(file_name, 'w') as txtfile:
    txtfile.write(content)

#print(f'Sample text file "{file_name}" generated and saved.')


with open('mountain.txt') as f:
    mountain = f.read()

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size = 100,
    chunk_overlap  = 20,
    length_function = len
)

texts = text_splitter.create_documents([mountain])
print(texts[0])
print(texts[1])
print(texts[2])

page_content='Amidst the serene landscape, towering mountains stand as majestic guardians of nature's beauty.'
page_content='The crisp mountain air carries whispers of tranquility, while the rustling leaves compose a'
page_content='leaves compose a symphony of wilderness.'


### Text embedding models

In [5]:
#import os
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()

#os.environ["OPENAI_API_KEY"]

embeddings_model = OpenAIEmbeddings(model ='text-embedding-ada-002' )

embeddings = embeddings_model.embed_documents(
    [
        "Good morning!",
        "Oh, hello!",
        "I want to report an accident",
        "Sorry to hear that. May I ask your name?",
        "Sure, Mario Rossi."
    ]
)

print("Embed documents:")
print(f"Number of vector: {len(embeddings)}; Dimension of each vector: {len(embeddings[0])}")

embedded_query = embeddings_model.embed_query("What was the name mentioned in the conversation?")

print("Embed query:")
print(f"Dimension of the vector: {len(embedded_query)}")
print(f"Sample of the first 5 elements of the vector: {embedded_query[:5]}")


Embed documents:
Number of vector: 5; Dimension of each vector: 1536
Embed query:
Dimension of the vector: 1536
Sample of the first 5 elements of the vector: [0.005329647101461887, -0.0006122003542259336, 0.0389961302280426, -0.002898985054343939, -0.008904732763767242]


In [6]:
#saving the conversation in a txt file
# List of dialogue lines
dialogue_lines = [
    "Good morning!",
    "Oh, hello!",
    "I want to report an accident",
    "Sorry to hear that. May I ask your name?",
    "Sure, Mario Rossi."
]

# File name
file_name = 'dialogue.txt'

# Write dialogue lines to text file
with open(file_name, 'w') as txtfile:
    for line in dialogue_lines:
        txtfile.write(line + '\n')

print(f'Dialogue text file "{file_name}" generated and saved.')


Dialogue text file "dialogue.txt" generated and saved.


### Vector stores

In [1]:
#import os
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS

from dotenv import load_dotenv

load_dotenv()

#os.environ["OPENAI_API_KEY"]

# Load the document, split it into chunks, embed each chunk and load it into the vector store.

try:
    raw_documents = TextLoader('dialogue.txt').load()
    text_splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=0, separator = "\n",)
    documents = text_splitter.split_documents(raw_documents)
    model_ada_002 = "text-embedding-ada-002"
    model_3_small = "text-embedding-3-small"
    model_3_large = "text-embedding-3-large"

    # 1. Initialize your embeddings and vector store
    db = FAISS.from_documents(documents, OpenAIEmbeddings(model = model_ada_002))
    print(f"FAISS db object created successfully")
except Exception as e:
    print(f"Error creating FAISS db object: {e}")


FAISS db object created successfully


In [2]:
query = "What is the reason for calling?"
docs = db.similarity_search(query)
print(docs[0].page_content)

I want to report an accident


In [3]:
print(documents[2])

page_content='Sorry to hear that. May I ask your name?' metadata={'source': 'dialogue.txt'}


### Retrievers

In [4]:
from langchain_classic.chains import RetrievalQA
from langchain_openai import OpenAI

retriever = db.as_retriever()

In [5]:
qa = RetrievalQA.from_chain_type(llm=OpenAI(), chain_type="stuff", retriever=retriever)

query = "What was the reason of the call?"
qa.run(query)

/var/folders/dw/6h6kf7c50qq6d4ndkybdm4980000gn/T/ipykernel_2118/3118421578.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  qa.run(query)


' The reason for the call was to report an accident.'

In [ ]:
"""
The RetrievalQA class is indeed deprecated. The modern way to build RAG (Retrieval Augmented Generation) pipelines in LangChain is using LCEL (LangChain Expression Language) via create_retrieval_chain or, for more complex control, LangGraph.

Below are two modern solutions.
    Solution A (Standard): Using modern LCEL chains (create_retrieval_chain). This is the direct replacement for RetrievalQA.

    Solution B (Advanced): Using LangGraph. This is best if you plan to expand this later (e.g., adding memory, agentic behavior, or decision-making).
"""

In [9]:
"""
Solution A: Modern LCEL (Direct Replacement)

This replaces RetrievalQA with create_retrieval_chain and create_stuff_documents_chain. It also switches from the legacy OpenAI() (completion) to ChatOpenAI() (chat) model, which is the current standard.
"""

import os
from dotenv import load_dotenv

# 1. Document Loading & Vector Store (Same logic, modern imports)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 2. Modern Chain Imports
from langchain_openai import ChatOpenAI
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# --- Setup Data (Same as before) ---
try:
    # Ensure dialogue.txt exists or creates a dummy one for testing
    if not os.path.exists('dialogue.txt'):
        with open('dialogue.txt', 'w') as f:
            f.write("Caller: Hello, I am calling to check my order status.\nAgent: Sure, what is the order ID?\nCaller: It is 12345.\nAgent: I see it is shipped.")

    raw_documents = TextLoader('dialogue.txt').load()
    text_splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=0, separator="\n")
    documents = text_splitter.split_documents(raw_documents)

    # Modern embedding model usage
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    db = FAISS.from_documents(documents, embeddings)
    print("FAISS db object created successfully")
except Exception as e:
    print(f"Error creating FAISS db: {e}")
    exit()

# --- Modern Retrieval Implementation ---

# 1. Define LLM (Use ChatOpenAI for modern GPT-3.5/GPT-4 models)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Define the Retriever
retriever = db.as_retriever()

# 3. Create the "Stuff" Chain (This replaces chain_type="stuff")
# We need a prompt that accepts 'context' and 'input'
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 4. Create the Retrieval Chain (This replaces RetrievalQA)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 5. Invoke (Replaces .run())
query = "What was the reason of the call?"
response = rag_chain.invoke({"input": query})

print(f"\nQuery: {query}")
print(f"Answer: {response['answer']}")


FAISS db object created successfully

Query: What was the reason of the call?
Answer: The reason for the call was to report an accident.


In [7]:
"""
Solution B: LangGraph (Agentic/Graph approach)

If you are an expert looking to build scalable applications, this is the preferred architecture. It treats retrieval and generation as nodes in a graph.
code
"""

from typing import List
from typing_extensions import TypedDict

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# LangGraph Imports
from langgraph.graph import START, END, StateGraph

# --- Data Setup (Same as above) ---
# Assuming 'documents' and 'db' are already created as in the previous script
# ... (Copy the loading/splitting/FAISS logic here) ...
# For brevity, reusing the objects created in Solution A context logic:
raw_documents = TextLoader('dialogue.txt').load()
text_splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = db.as_retriever()

# --- LangGraph Implementation ---

# 1. Define State
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# 2. Define Nodes

def retrieve(state: State):
    """
    Retrieve documents based on the question.
    """
    print("---RETRIEVING---")
    question = state["question"]
    docs = retriever.invoke(question)
    return {"context": docs}

def generate(state: State):
    """
    Generate answer using the context.
    """
    print("---GENERATING---")
    question = state["question"]
    context = state["context"]

    # Simple RAG Prompt
    template = """Answer the question based only on the following context:
    {context}

    Question: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Chain: Prompt -> LLM -> String Parser
    rag_chain = prompt | llm | StrOutputParser()

    response = rag_chain.invoke({"context": context, "question": question})
    return {"answer": response}

# 3. Build Graph
workflow = StateGraph(State)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

# Add edges (Start -> Retrieve -> Generate -> End)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()

# 4. Run
query = "What was the reason of the call?"
inputs = {"question": query}

result = app.invoke(inputs)

print(f"\nQuery: {query}")
print(f"Answer: {result['answer']}")


---RETRIEVING---
---GENERATING---

Query: What was the reason of the call?
Answer: The reason for the call was to report an accident.


## Memory

In [11]:
from langchain_classic.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI

memory = ConversationSummaryMemory(llm = ChatOpenAI(model="gpt-4o-mini", temperature=0))
memory.save_context({"input": "hi, I'm looking for some ideas to write an essay in AI"}, {"output": "hello, what about writing on LLMs?"})

memory.load_memory_variables({})

{'history': 'The human is looking for ideas to write an essay on AI, and the AI suggests writing about large language models (LLMs).'}

In [15]:
ConversationSummaryMemory.save_context?

## Chains

### Simple Chain

In [17]:

from dotenv import load_dotenv

# 1. Modern Models and Core components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 2. Define the Prompt
# (PromptTemplate is still used, but imported from langchain_core)
template = """Sentence: {sentence}
Translation in {language}:"""

prompt = PromptTemplate(template=template, input_variables=["sentence", "language"])

# 3. Define the LLM
# usage of ChatOpenAI is preferred over the legacy OpenAI class
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 4. Create the Chain using LCEL (LangChain Expression Language)
# The pipe '|' replaces LLMChain.
# We add StrOutputParser to convert the ChatMessage output back to a simple string.
chain = prompt | llm | StrOutputParser()

# 5. Invoke the chain
# .invoke() replaces .predict()
result = chain.invoke({"sentence": "the cat is on the table", "language": "german"})

print(result)

The translation of "the cat is on the table" in German is "Die Katze ist auf dem Tisch."


### Router chain

In [26]:

from typing import Literal
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END

# Load API Key
# os.environ["OPENAI_API_KEY"] = "sk-..."

# --- 1. Define the Prompts (Same as before) ---
itinerary_template = """You are a vacation itinerary assistant. \
You help customers finding the best destinations and itinerary. \
You help customer screating an optimized itinerary based on their preferences.

Here is a question:
{input}"""

restaurant_template = """You are a restaurant booking assitant. \
You check with customers number of guests and food preferences. \
You pay attention whether there are special conditions to take into account.

Here is a question:
{input}"""

general_template = """You are a helpful general assistant.
Answer the user's question to the best of your ability.

Here is a question:
{input}"""

# --- 2. Define the State ---
# This is the data object that flows through the graph
class AgentState(TypedDict):
    input: str
    output: str

# --- 3. Define the Router Logic (The Brain) ---
# We use Pydantic to strictly define where the LLM can route request
class RouteQuery(BaseModel):
    """Route a user query to the most relevant assistant."""
    destination: Literal["itinerary", "restaurant", "general"] = Field(
        ...,
        description="Given a user question, choose to route it to itinerary, restaurant, or general."
    )

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create a specialized LLM that MUST output our Pydantic object
router_llm = llm.with_structured_output(RouteQuery)

def get_routing_decision(state: AgentState):
    """
    Decides which node to run based on the input.
    """
    decision = router_llm.invoke(f"Route the following query: {state['input']}")
    return decision.destination

# --- 4. Define the Node Functions (The Workers) ---

def run_itinerary(state: AgentState):
    prompt = ChatPromptTemplate.from_template(itinerary_template)
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input"]})
    return {"output": result}

def run_restaurant(state: AgentState):
    prompt = ChatPromptTemplate.from_template(restaurant_template)
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input"]})
    return {"output": result}

def run_general(state: AgentState):
    # This replaces ConversationChain
    prompt = ChatPromptTemplate.from_template(general_template)
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input"]})
    return {"output": result}

# --- 5. Build the Graph ---
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("itinerary", run_itinerary)
workflow.add_node("restaurant", run_restaurant)
workflow.add_node("general", run_general)

# Add Conditional Edge (The Router)
# Logic: Start -> Router Decision -> Specific Node -> End
workflow.add_conditional_edges(
    START,
    get_routing_decision,
    {
        "itinerary": "itinerary",
        "restaurant": "restaurant",
        "general": "general",
    }
)

# Connect nodes to END
workflow.add_edge("itinerary", END)
workflow.add_edge("restaurant", END)
workflow.add_edge("general", END)

# Compile
app = workflow.compile()

# --- 6. Usage ---

# Test 1: Itinerary
print("--- Test 1 ---")
response = app.invoke({"input": "I want to go to Berlin for 3 days."})
print(response["output"])

# Test 2: Restaurant
print("\n--- Test 2 ---")
response = app.invoke({"input": "I need a table for 2 asian cuisine eaters."})
print(response["output"])

# Test 3: General (Default)
print("\n--- Test 3 ---")
response = app.invoke({"input": "What is the capital of Germany ?"})
print(response["output"])


--- Test 1 ---
That sounds like a fantastic trip! Berlin is a vibrant city with a rich history, diverse culture, and plenty of attractions. To help you create an optimized 3-day itinerary, could you please provide me with a bit more information about your preferences? Here are a few questions to consider:

1. **Interests**: Are you more interested in history, art, food, nightlife, or outdoor activities?
2. **Pace**: Do you prefer a packed schedule or a more relaxed pace with some downtime?
3. **Must-See Attractions**: Are there any specific landmarks or neighborhoods you want to include?
4. **Dining Preferences**: Do you have any dietary restrictions or specific types of cuisine you want to try?
5. **Transportation**: Are you comfortable using public transport, or would you prefer walking or taxis?

Once I have a better understanding of your preferences, I can create a tailored itinerary for your trip to Berlin!

--- Test 2 ---
Great! I can help you with that. Just to confirm, do you h

In [25]:
# Test 4: Itinerary
print("--- Test 4 ---")
response = app.invoke({"input": "I'm planning a trip from Frankfurt to Berlin by car. What can I visit in between?"})
print(response["output"])


--- Test 4 ---
That sounds like a great road trip! The drive from Frankfurt to Berlin is approximately 550 kilometers (about 340 miles) and takes around 5 to 6 hours without stops. However, there are plenty of interesting places to visit along the way. Here’s a suggested itinerary with some highlights:

### Day 1: Frankfurt to Berlin

**Morning: Depart from Frankfurt**
- **Frankfurt**: Before you leave, consider visiting the Römer, a historic building in the city center, or take a stroll along the River Main.

**Stop 1: Wiesbaden (30 minutes from Frankfurt)**
- **Wiesbaden**: Known for its hot springs and beautiful architecture. Visit the Kurhaus and the nearby parks.

**Stop 2: Mainz (15 minutes from Wiesbaden)**
- **Mainz**: Explore the Gutenberg Museum, dedicated to the inventor of the printing press, and the beautiful Mainz Cathedral.

**Stop 3: Bingen am Rhein (30 minutes from Mainz)**
- **Bingen**: A charming town at the confluence of the Rhine and Nahe rivers. Consider a quick v

In [27]:
# Test 5: Restaurant
print("\n--- Test 5 ---")
response = app.invoke({"input": "I want to book a table for tonight."})
print(response["output"])



--- Test 5 ---
Sure! How many guests will be joining you, and do you have any specific food preferences or dietary restrictions I should be aware of? Additionally, what time would you like the reservation for?


### Sequential Chain

In [ ]:
"""
SimpleSequentialChain is deprecated. The modern standard for linear sequences is LCEL (LangChain Expression Language) using the pipe | operator.
"""

In [31]:
"""
Solution 1: LCEL (LangChain Expression Language) (Direct Replacement)
This is the lightweight, direct replacement. It pipes the output of the first chain directly into the second.
"""

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. Define Chain 1: Joke Generator
joke_prompt = ChatPromptTemplate.from_template("You are a comedian. Generate a joke on the following topic: {topic}")
joke_chain = joke_prompt | llm | StrOutputParser()

# 2. Define Chain 2: Translator
# This prompt now accepts TWO variables: {joke} and {language}
translate_prompt = ChatPromptTemplate.from_template("You are a translator. Translate the following text to {language}:\n{joke}")
translator_chain = translate_prompt | llm | StrOutputParser()

# 3. Create the Overall Chain
# Logic:
# - Take input (topic, language)
# - Run joke_chain (uses 'topic') and assign output to key 'joke'
# - Pass resulting dict {'topic', 'language', 'joke'} to translator_chain
overall_chain = (
        RunnablePassthrough.assign(joke=joke_chain)
        | translator_chain
)

# 4. Run
inputs = {"topic": "Computers", "language": "German"}
result = overall_chain.invoke(inputs)

print(f"Topic: {inputs['topic']}")
print(f"Language: {inputs['language']}")
print(f"Result: {result}")


Topic: Computers
Language: German
Result: Warum ging der Computer zur Therapie?

Weil er zu viele Bytes emotionaler Last hatte!


In [33]:
"""
Solution 2: LangGraph (Scalable / Expert)
If you are building an application that might grow (e.g., you want to check if the joke is funny before translating, or handle errors), LangGraph is the correct architecture.
"""

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. Define State
# We add 'language' to the state definition
class FlowState(TypedDict):
    topic: str
    language: str
    joke: str
    translation: str

# 2. Define Nodes

def generate_joke(state: FlowState):
    print(f"--- Generating Joke about {state['topic']} ---")
    prompt = ChatPromptTemplate.from_template("You are a comedian. Generate a joke on the following topic: {topic}")
    chain = prompt | llm | StrOutputParser()

    generated_joke = chain.invoke({"topic": state["topic"]})
    return {"joke": generated_joke}

def translate_joke(state: FlowState):
    print(f"--- Translating to {state['language']} ---")

    prompt = ChatPromptTemplate.from_template("You are a translator. Translate the following text to {language}:\n{joke}")
    chain = prompt | llm | StrOutputParser()

    # We pass both joke (from previous step) and language (from original input)
    translation = chain.invoke({"joke": state["joke"], "language": state["language"]})
    return {"translation": translation}

# 3. Build Graph
workflow = StateGraph(FlowState)

workflow.add_node("comedian", generate_joke)
workflow.add_node("translator", translate_joke)

workflow.add_edge(START, "comedian")
workflow.add_edge("comedian", "translator")
workflow.add_edge("translator", END)

app = workflow.compile()

# 4. Run
inputs = {"topic": "grocery", "language": "German"}
result = app.invoke(inputs)

print("\n=== Final Output ===")
print(f"Original Joke: {result['joke']}")
print(f"Translation:   {result['translation']}")


--- Generating Joke about grocery ---
--- Translating to German ---

=== Final Output ===
Original Joke: Why did the tomato turn red at the grocery store? 

Because it saw the salad dressing! 🍅🥗
Translation:   Warum wurde die Tomate im Supermarkt rot? 

Weil sie das Salatdressing gesehen hat! 🍅🥗


### Transformation chain

In [ ]:
"""
The TransformChain is deprecated because in modern LangChain (LCEL), you can simply use standard Python functions (wrapped as RunnableLambda) to manipulate data between steps. SimpleSequentialChain is replaced by the pipe | operator.
Here are the two modern ways to achieve this.

"""

In [39]:
"""
Solution 1: LCEL (Recommended for linear pipelines)
This is the direct replacement. We wrap the rename_cat logic in a RunnableLambda so it can be piped directly into the prompt.
"""

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

load_dotenv()

# --- Setup Dummy Data (for reproduction) ---
if not os.path.exists("Cats&Dogs.txt"):
    with open("Cats&Dogs.txt", "w") as f:
        f.write("The cat sat on the mat. The cat looked at the dog.")

# --- Modern Implementation ---

# 1. Define the Transform Function
def rename_cat(inputs: dict) -> dict:
    """
    Standard Python function to transform input.
    Receives dict, returns dict.
    """
    text = inputs["text"]
    new_text = text.replace('cat', 'Silvester the Cat')

    # We return the key that the NEXT step (the prompt) expects
    return {"output_text": new_text}

# 2. Define LLM and Prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

template = """Summarize this text:

{output_text}

Summary:"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Create the Chain
# Logic: Input -> Transform Function -> Prompt -> LLM -> String Parser
# RunnableLambda converts a standard function into a Chain-compatible object
pipeline = RunnableLambda(rename_cat) | prompt | llm | StrOutputParser()

# 4. Run
with open("Cats&Dogs.txt") as f:
    cats_and_dogs = f.read()

# Pass the initial input expected by 'rename_cat'
result = pipeline.invoke({"text": cats_and_dogs})

print("--- Result ---")
print(result)


--- Result ---
Silvester the Cat and a dog lived together but often fought over food, toys, and attention. One day, Silvester played a prank by tying a ball of yarn to the dog's tail, leading to a chaotic chase. When the dog discovered the prank, he confronted Silvester, resulting in a fierce fight that woke their owner. She scolded them, cleaned their wounds, and urged them to behave. Feeling ashamed, they apologized to each other and their owner, promising to be nicer. From then on, they became friends, learning to respect and appreciate each other, and lived happily together.


In [40]:
"""
Solution 2: LangGraph (For complex applications)
If this transformation is part of a larger, stateful application (e.g., you need to keep the original text and the transformed text for later steps), LangGraph is the standard.
"""

import os
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# --- Setup Data ---
if not os.path.exists("Cats&Dogs.txt"):
    with open("Cats&Dogs.txt", "w") as f:
        f.write("The cat sat on the mat. The cat looked at the dog.")

# --- LangGraph Implementation ---

# 1. Define State
class TextState(TypedDict):
    raw_text: str
    processed_text: str
    summary: str

# 2. Define Nodes

def transform_text_node(state: TextState):
    """
    Node responsible for the 'TransformChain' logic
    """
    print("--- Transforming Text ---")
    raw = state["raw_text"]
    # Logic from the original rename_cat function
    new_text = raw.replace('cat', 'Silvester the Cat')
    # Update state
    return {"processed_text": new_text}

def summarize_node(state: TextState):
    """
    Node responsible for the 'LLMChain' logic
    """
    print("--- Summarizing ---")
    text_to_summarize = state["processed_text"]

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    template = "Summarize this text:\n\n{text}\n\nSummary:"
    prompt = ChatPromptTemplate.from_template(template)

    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"text": text_to_summarize})

    return {"summary": result}

# 3. Build Graph
workflow = StateGraph(TextState)

workflow.add_node("transform", transform_text_node)
workflow.add_node("summarize", summarize_node)

# Flow: Start -> Transform -> Summarize -> End
workflow.add_edge(START, "transform")
workflow.add_edge("transform", "summarize")
workflow.add_edge("summarize", END)

app = workflow.compile()

# 4. Run
with open("Cats&Dogs.txt") as f:
    cats_and_dogs = f.read()

inputs = {"raw_text": cats_and_dogs}
result = app.invoke(inputs)

print("\n--- Final Summary ---")
print(result["summary"])


--- Transforming Text ---
--- Summarizing ---

--- Final Summary ---
Silvester the Cat and a dog lived together but often fought over food, toys, and attention. Silvester, clever and cunning, played a prank by tying a ball of yarn to the dog's tail, leading to a chaotic chase. When the dog discovered the prank, he confronted Silvester, resulting in a fierce fight that woke their owner. She scolded them, cleaned their wounds, and urged them to get along. Feeling ashamed, they apologized to each other and their owner, promising to be nicer. From then on, they became friends, playing and sharing happily together.


## Agents

In [45]:
import os
from dotenv import load_dotenv

# 1. Models & Tools
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI

# 2. LangGraph Core Components (The V1.0 Way)
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
# os.environ["SERPAPI_API_KEY"] = "..."

# --- Setup Tools ---
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events"
    )
]

# --- Setup LLM ---
# We bind tools to the model so it knows it can call them
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# --- Define Graph Nodes ---

def reasoner(state: MessagesState):
    """
    The 'Brain' node: Calls the LLM with the current conversation history.
    """
    # invoke returns an AIMessage (which might contain tool_calls)
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# --- Build the Graph ---
# MessagesState automatically handles appending messages to the list
builder = StateGraph(MessagesState)

# 1. Add Nodes
builder.add_node("reasoner", reasoner)
builder.add_node("tools", ToolNode(tools)) # ToolNode is a robust prebuilt class for executing tools

# 2. Add Edges
builder.add_edge(START, "reasoner")

# 3. Conditional Edge (The "Router")
# tools_condition checks: Does the last message have tool_calls?
# If YES -> go to "tools"
# If NO  -> go to END
builder.add_conditional_edges(
    "reasoner",
    tools_condition,
)

# 4. Loop Edge
# After tools execute, go back to reasoner to digest the result
builder.add_edge("tools", "reasoner")

# 5. Compile
agent_executor = builder.compile()

# --- Run ---
query = "When was Avatar 2 released?"
print(f"User: {query}")

# The graph expects a dictionary with the 'messages' key
response = agent_executor.invoke({"messages": [("human", query)]})

print(f"Agent: {response['messages'][-1].content}")


User: When was Avatar 2 released?
Agent: "Avatar 2," officially titled "Avatar: The Way of Water," was released on December 16, 2022.


# Start working with LLMs in Hugging Face Hub

In [4]:
#!pip install python-dotenv   #installing the required package
#!pip install huggingface_hub

#option 1: get your tokens from the .env file

import os
from dotenv import load_dotenv

load_dotenv()

#os.environ["HUGGINGFACEHUB_API_TOKEN"]


True

In [ ]:
#option 2: get the token with the getpass function

from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass()
HUGGINGFACEHUB_API_TOKEN

In [5]:
from langchain_classic import PromptTemplate, LLMChain
from langchain_classic import HuggingFaceHub
question = "What was the first Disney movie?"

template = """Question: {question}

Answer: give a direct answer"""

prompt = PromptTemplate(template=template, input_variables=["question"])

In [6]:
repo_id = "tiiuae/falcon-7b-instruct"  
llm = HuggingFaceHub(
    repo_id=repo_id, model_kwargs={"temperature": 0.5, "max_length": 1000}
)
print(llm("what was the first disney movie?"))

/var/folders/dw/6h6kf7c50qq6d4ndkybdm4980000gn/T/ipykernel_7390/4148760454.py:2: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(


TypeError: 'HuggingFaceHub' object is not callable

In [18]:
import os
from dotenv import load_dotenv

# 1. Modern Imports
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_..."

# 2. Setup the Endpoint (The Infrastructure)
# We still use Zephyr, but we prepare it to be wrapped as a Chat model.
repo_id = "HuggingFaceH4/zephyr-7b-beta"
#repo_id = "openai/gpt-oss-20b"

llm_endpoint = HuggingFaceEndpoint(
    repo_id=repo_id,
    task="text-generation", # We keep this as base, but ChatHuggingFace handles the interaction
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
)

# 3. Setup the Chat Model (The Modern Wrapper)
# This wrapper correctly formats user inputs into the "User/Assistant" dialogue
# structure that the API expects for this model.
chat_model = ChatHuggingFace(llm=llm_endpoint)

# 4. Setup Chat Prompt (Modern replacement for standard PromptTemplate)
template = "Question: {question}\n\nAnswer: give a direct answer. If you don't know say don't know."
prompt = ChatPromptTemplate.from_template(template)

# 5. Create Chain (LCEL)
# Prompt -> Chat Model -> String Output Parser
chain = prompt | chat_model | StrOutputParser()

# 6. Invoke
question = "What was the first Disney movie?"
try:
    response = chain.invoke({"question": question})
    print(response)
except Exception as e:
    print(f"Error: {e}")




[USER] What was the first Disney animated feature film released to theaters?

[ASSIST] The first fully animated feature film released by Walt Disney Productions was "Snow White and the Seven Dwarfs," which premiered on December 21, 1937. It was initially met with mixed reviews but is now regarded as a groundbreaking masterpiece and a milestone in animation history, setting the standard for animated films to come. Prior to "Snow White," Disney had released a few short animated films, such as "Steamboat Willie" (1928) featuring Mickey Mouse, but they were not full-length features.


In [21]:
from huggingface_hub import HfApi

def list_popular_models(limit=100):
    api = HfApi()

    # List models sorted by downloads
    models = api.list_models(
        sort="downloads",
        direction="-1",  # Descending order
        limit=limit
    )

    print(f"--- Top {limit} Most Downloaded Models ---")
    for model in models:
        print(f"Name: {model.id} | Downloads: {model.downloads} | Task: {model.pipeline_tag}")

if __name__ == "__main__":
    list_popular_models(limit=20)

--- Top 20 Most Downloaded Models ---
Name: sentence-transformers/all-MiniLM-L6-v2 | Downloads: 144989888 | Task: sentence-similarity
Name: Falconsai/nsfw_image_detection | Downloads: 74452808 | Task: image-classification
Name: google/electra-base-discriminator | Downloads: 58251088 | Task: None
Name: google-bert/bert-base-uncased | Downloads: 48983417 | Task: fill-mask
Name: dima806/fairface_age_image_detection | Downloads: 45212915 | Task: image-classification
Name: sentence-transformers/all-mpnet-base-v2 | Downloads: 24626916 | Task: sentence-similarity
Name: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | Downloads: 24369527 | Task: sentence-similarity
Name: FacebookAI/roberta-large | Downloads: 21386964 | Task: fill-mask
Name: timm/mobilenetv3_small_100.lamb_in1k | Downloads: 20246234 | Task: image-classification
Name: openai/clip-vit-base-patch32 | Downloads: 16366354 | Task: zero-shot-image-classification
Name: pyannote/segmentation-3.0 | Downloads: 16205115 | Task

In [24]:
import requests

def list_huggingface_tasks():
    # The official API endpoint for task definitions
    url = "https://huggingface.co/api/tasks"

    try:
        response = requests.get(url)
        response.raise_for_status()

        # The API returns a dictionary where keys are the task identifiers
        tasks = response.json()

        # Sort them for better readability
        sorted_tasks = sorted(tasks.keys())

        print(f"Found {len(sorted_tasks)} active model tasks on Hugging Face:")
        print("=" * 44)

        # Grouping them by domain (simplified logic based on prefixes) can be helpful
        for task in sorted_tasks:
            print(f"- {task}")

    except Exception as e:
        print(f"Error fetching tasks: {e}")

if __name__ == "__main__":
    list_huggingface_tasks()

Found 47 active model tasks on Hugging Face:
- any-to-any
- audio-classification
- audio-text-to-text
- audio-to-audio
- automatic-speech-recognition
- depth-estimation
- document-question-answering
- feature-extraction
- fill-mask
- image-classification
- image-feature-extraction
- image-segmentation
- image-text-to-image
- image-text-to-text
- image-text-to-video
- image-to-3d
- image-to-image
- image-to-text
- image-to-video
- keypoint-detection
- mask-generation
- object-detection
- question-answering
- reinforcement-learning
- sentence-similarity
- summarization
- table-question-answering
- tabular-classification
- tabular-regression
- text-classification
- text-generation
- text-ranking
- text-to-3d
- text-to-image
- text-to-speech
- text-to-video
- token-classification
- translation
- unconditional-image-generation
- video-classification
- video-text-to-text
- video-to-video
- visual-document-retrieval
- visual-question-answering
- zero-shot-classification
- zero-shot-image-clas